In [1]:
from utils import *

In [3]:
# Create (or update) the data store.
documents = load_documents()
chunks = split_documents(documents)
add_to_chroma(chunks)

Number of existing documents in DB: 0
👉 Adding new documents: 265


In [2]:
clear_database()

In [5]:
from utils import *

def doc_id():
    loader = DirectoryLoader(DATA_PATH, glob="*.md")
    documents = loader.load()
    return [doc.lc_attributes for doc in documents]

In [6]:
doc_id()

[{}]

In [3]:
from llama_index.core import SimpleDirectoryReader

import os

input_files = [
    os.path.join("data", "alice_in_wonderland.md")
]

documents = SimpleDirectoryReader(input_files=input_files).load_data()

for doc in documents:
    print(doc.doc_id)

8095d67c-0bc9-4d93-aae0-74e58fe6e6e2


In [ ]:
from llama_index.core import VectorStoreIndex, StorageContext, Settings
from llama_index.vector_stores.milvus import MilvusVectorStore
from llama_index.llms.ollama import Ollama
from llama_index.embeddings.ollama import OllamaEmbedding

embed_model = OllamaEmbedding("bge-m3")
llm_model = Ollama("qwen")

Settings.embed_model = embed_model
Settings.llm = llm_model

milvus_uri = os.path.join("milvus", "milvus_demo.db")

vector_store = MilvusVectorStore(uri=milvus_uri, dim=1536, overwrite=True)
storage_context = StorageContext.from_defaults(vector_store=vector_store)
index = VectorStoreIndex.from_documents(documents, storage_context=storage_context, show_progress=True)
index

In [1]:
from pymilvus import connections, db

conn = connections.connect(host="127.0.0.1", port=19530)

database = db.create_database("my_database")

In [1]:
from pymilvus import MilvusClient

# 1. Set up a Milvus client
client = MilvusClient(
    uri="http://localhost:19530"
)

# client.create_database('phuc_database')
# 2. Create a collection
# client.create_collection(
#     collection_name="quick_setup",
#     dimension=5,
#     metric_type="IP"
# )

In [2]:
client.list_collections()
client.describe_collection("phuc_collection")

{'collection_name': 'phuc_collection',
 'auto_id': True,
 'num_shards': 1,
 'description': '',
 'fields': [{'field_id': 100,
   'name': 'id',
   'description': '',
   'type': <DataType.INT64: 5>,
   'params': {},
   'auto_id': True,
   'is_primary': True},
  {'field_id': 101,
   'name': 'vector',
   'description': '',
   'type': <DataType.FLOAT_VECTOR: 101>,
   'params': {'dim': 10}}],
 'aliases': [],
 'collection_id': 453972108735571422,
 'consistency_level': 0,
 'properties': {},
 'num_partitions': 1,
 'enable_dynamic_field': False}

In [2]:
data=[
    {"id": 0, "vector": [0.3580376395471989, -0.6023495712049978, 0.18414012509913835, -0.26286205330961354, 0.9029438446296592], "color": "pink_8682"},
    {"id": 1, "vector": [0.19886812562848388, 0.06023560599112088, 0.6976963061752597, 0.2614474506242501, 0.838729485096104], "color": "red_7025"},
    {"id": 2, "vector": [0.43742130801983836, -0.5597502546264526, 0.6457887650909682, 0.7894058910881185, 0.20785793220625592], "color": "orange_6781"},
    {"id": 3, "vector": [0.3172005263489739, 0.9719044792798428, -0.36981146090600725, -0.4860894583077995, 0.95791889146345], "color": "pink_9298"},
    {"id": 4, "vector": [0.4452349528804562, -0.8757026943054742, 0.8220779437047674, 0.46406290649483184, 0.30337481143159106], "color": "red_4794"},
    {"id": 5, "vector": [0.985825131989184, -0.8144651566660419, 0.6299267002202009, 0.1206906911183383, -0.1446277761879955], "color": "yellow_4222"},
    {"id": 6, "vector": [0.8371977790571115, -0.015764369584852833, -0.31062937026679327, -0.562666951622192, -0.8984947637863987], "color": "red_9392"},
    {"id": 7, "vector": [-0.33445148015177995, -0.2567135004164067, 0.8987539745369246, 0.9402995886420709, 0.5378064918413052], "color": "grey_8510"},
    {"id": 8, "vector": [0.39524717779832685, 0.4000257286739164, -0.5890507376891594, -0.8650502298996872, -0.6140360785406336], "color": "white_9381"},
    {"id": 9, "vector": [0.5718280481994695, 0.24070317428066512, -0.3737913482606834, -0.06726932177492717, -0.6980531615588608], "color": "purple_4976"}
]

res = client.insert(
    collection_name="quick_setup",
    data=data
)

print(res)



{'insert_count': 10, 'ids': [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]}


In [4]:
import json

res = client.search(
    collection_name="quick_setup", # Replace with the actual name of your collection
    # Replace with your query vector
    data=[[0.3580376395471989, -0.6023495712049978, 0.18414012509913835, -0.26286205330961354, 0.9029438446296592]],
    limit=5, # Max. number of search results to return
    search_params={"metric_type": "IP", "params": {}} # Search parameters
)

result = json.dumps(res, indent=4)
print(result)


[
    [
        {
            "id": 0,
            "distance": 1.4093276262283325,
            "entity": {}
        },
        {
            "id": 4,
            "distance": 0.9902133941650391,
            "entity": {}
        },
        {
            "id": 1,
            "distance": 0.8519943356513977,
            "entity": {}
        },
        {
            "id": 5,
            "distance": 0.797234296798706,
            "entity": {}
        },
        {
            "id": 2,
            "distance": 0.5928734540939331,
            "entity": {}
        }
    ]
]


In [15]:
import json

res = client.search(
    collection_name="quick_setup", 
    data=[[2, -4, 3, 2, 1]], 
    limit=5, 
    output_fields=["color"],
    search_params={"metric_type": "IP", "params": {}})

result = json.dumps(res, indent=4)
print(result)

[
    [
        {
            "id": 4,
            "distance": 8.091014862060547,
            "entity": {
                "color": "red_4794"
            }
        },
        {
            "id": 5,
            "distance": 7.2160444259643555,
            "entity": {
                "color": "yellow_4222"
            }
        },
        {
            "id": 2,
            "distance": 6.8378801345825195,
            "entity": {
                "color": "orange_6781"
            }
        },
        {
            "id": 7,
            "distance": 5.472618579864502,
            "entity": {
                "color": "grey_8510"
            }
        },
        {
            "id": 0,
            "distance": 4.055113792419434,
            "entity": {
                "color": "pink_8682"
            }
        }
    ]
]
